# 05. Perceptron

Linear classifier without probabilities (uses decision_function).
Tune L2 strength `alpha` with stratified CV (F1); keep max_iter=1000.

In [1]:
import sys
sys.path.append("..")

from sklearn.linear_model import Perceptron

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult("train"))
X_test, y_test = split_features_target(load_adult("test"))
print(X.shape, y.mean())


(32561, 12) 0.2408095574460244


In [2]:
# Find the best Hyperparameter
candidates = [1e-4, 1e-3, 1e-2, 1e-1]
selection_metric = "f1"

cv_by_alpha = {}
best_alpha = None
best_score = -1.0

for alpha in candidates:
    model = Perceptron(
        penalty="l2",
        alpha=alpha,
        max_iter=1000,
        random_state=42 
    )
    pipe = make_model_pipeline(model)
    cv = cross_val_scores(pipe, X, y)
    cv_by_alpha[alpha] = cv
    score = cv[selection_metric]["mean"]
    print(f"alpha={alpha} CV {selection_metric}={score:.4f} (±{cv[selection_metric]['std']:.4f})")

    if score > best_score:
        best_score = score
        best_alpha = alpha

print("selected alpha:", best_alpha)

alpha=0.0001 CV f1=0.5400 (±0.1623)
alpha=0.001 CV f1=0.5127 (±0.1270)
alpha=0.01 CV f1=0.3784 (±0.1364)
alpha=0.1 CV f1=0.2414 (±0.1579)
selected alpha: 0.0001


In [3]:
final_model = Perceptron(
        penalty="l2",
        alpha=best_alpha,
        max_iter=1000,
        random_state=42 
    )
cv = cv_by_alpha[best_alpha]
pipe2 = make_model_pipeline(final_model)
test = evaluate_on_test(pipe2, X, y, X_test, y_test)

path = save_results(
    "perceptron",
    cv,
    test,
    extra={
        "penalty": "l2",
        "alpha": best_alpha,
        "max_iter": 1000,
        "candidates": candidates,
        "selection_metric": selection_metric,
        "cv_by_alpha": {
            str(a): {m: v["mean"] for m, v in summary.items()}
            for a, summary in cv_by_alpha.items()
        },
    },
)

print("Test:", test["metrics"])
print("Confusion matrix:", test["confusion_matrix"])
print("saved:", path)

Test: {'accuracy': 0.798292488176402, 'balanced_accuracy': 0.7813899045708511, 'f1': 0.6370468611847923, 'roc_auc': 0.8750976842451261, 'average_precision': 0.6696564993435661}
Confusion matrix: [[10115, 2320], [964, 2882]]
saved: /Users/kang/Desktop/ML project/Estimating-Income/results/perceptron.json
